# 01_DIM_DATE

This notebook creates and loads the Date Dimension for the Data Warehouse.



# SETUP

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from delta import configure_spark_with_delta_pip
import ConnectionConfig as cc
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql.types import LongType, DateType, IntegerType, StringType, BooleanType

# Enable debugging
debugging_mode = True

# Initialize environment


In [8]:
cc.setupEnvironment()
spark = cc.startLocalCluster("DIM_DATE")

spark.getActiveSession()

Running in Docker environment.
Setting JAVA_HOME to /usr/lib/jvm/java-17-openjdk-arm64
Running in Docker: using pre-installed Spark + JARs


# Parameters

In [11]:
beginDate = '2020-09-11'
endDate = '2023-09-12'

# Generate date range

In [12]:
# EXTRACT

df_dates = spark.sql(f"""
SELECT
    explode(sequence(to_date('{beginDate}'), to_date('{endDate}'), interval 1 day)) AS full_date
""")

df_dates.createOrReplaceTempView("neededDates")


In [13]:
# DEBUG_CODE

if debugging_mode:
    spark.sql("SELECT * FROM neededDates").show(5)

+----------+
| full_date|
+----------+
|2020-09-11|
|2020-09-12|
|2020-09-13|
|2020-09-14|
|2020-09-15|
+----------+
only showing top 5 rows


# Create all typical dimension fields


In [14]:
# TRANSFORM

dimDate = spark.sql("""
SELECT
    full_date,
    dayofmonth(full_date) AS day,
    month(full_date) AS month,
    year(full_date) AS year,
    date_format(full_date, 'MMMM') AS month_name,
    date_format(full_date, 'EEEE') AS weekday_name,
    CASE
        WHEN dayofweek(full_date) IN (1,7) THEN true
        ELSE false
    END AS is_weekend
FROM neededDates
ORDER BY full_date
""")

In [15]:
# DEBUG_CODE

if debugging_mode:
    dimDate.show(5, truncate=False)

+----------+---+-----+----+----------+------------+----------+
|full_date |day|month|year|month_name|weekday_name|is_weekend|
+----------+---+-----+----+----------+------------+----------+
|2020-09-11|11 |9    |2020|September |Friday      |false     |
|2020-09-12|12 |9    |2020|September |Saturday    |true      |
|2020-09-13|13 |9    |2020|September |Sunday      |true      |
|2020-09-14|14 |9    |2020|September |Monday      |false     |
|2020-09-15|15 |9    |2020|September |Tuesday     |false     |
+----------+---+-----+----+----------+------------+----------+
only showing top 5 rows


# LOAD with Delta Identity

In [16]:
import shutil
import os

warehouse_path = spark.conf.get("spark.sql.warehouse.dir")
table_path = os.path.join(warehouse_path, "dim_date")

if os.path.exists(table_path):
    shutil.rmtree(table_path)

In [ ]:
# Drop table if exists
spark.sql("DROP TABLE IF EXISTS dim_date")

In [17]:
# CREATE dimDate table with surrogate key

DeltaTable.create(spark) \
    .tableName("dim_date") \
    .addColumn("date_sk", LongType(), nullable=False, generatedAlwaysAs=IdentityGenerator(0,1)) \
    .addColumn("full_date", DateType(), nullable=False) \
    .addColumn("day", IntegerType()) \
    .addColumn("month", IntegerType()) \
    .addColumn("year", IntegerType()) \
    .addColumn("month_name", StringType()) \
    .addColumn("weekday_name", StringType()) \
    .addColumn("is_weekend", BooleanType()) \
    .property("delta.feature.identityColumns", "supported") \
    .execute()

# Writing the data to a delta-file


In [18]:
# LOAD

dimDate.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("dim_date")

In [10]:
# Delete parquet file if it already exists
cc.delete_parquet_table("parquet/dim_date_parquet")

Deleted old parquet at spark-warehouse/dim_date_parquet


In [19]:
# Save as Parquet format

df = spark.read.format("delta").load("spark-warehouse/dim_date")

df.coalesce(1).write.format("parquet").save("./parquet/dim_date_parquet")

Validation

In [ ]:
# DEBUG_CODE

if debugging_mode:
    spark.read.format("delta").load("spark-warehouse/dim_date").select(count("*")).show()

cleanup

In [20]:
# CLEANUP

spark.stop()